# Build 2 — Verified Colab Model Backend

This notebook runs a small pretrained text model and creates a temporary backend for the supplied Apps Script interface.

The notebook now performs the complete connection sequence for you:

1. Load and test the model locally.
2. Start the local API.
3. Create a temporary HTTPS tunnel.
4. Wait for DNS readiness.
5. Test public health.
6. Send a real public model request.
7. Print the two exact values for Apps Script only after every test passes.

Do not construct or edit endpoint paths manually. Apps Script adds the required paths.

### Architecture

`Apps Script → deterministic validation → verified Colab backend → pretrained model → output checks → human review → approved action`

### Privacy

- Do not enter confidential, regulated, private, or identifying information.
- The temporary tunnel is suitable only for a classroom prototype.
- The randomly generated secret changes whenever the backend is relaunched.
- Keep this runtime connected while using the Apps Script application.


## 1. Start with a fresh Colab runtime

For the cleanest launch select **Runtime → Restart session** before choosing **Run all**.

A T4 GPU is recommended. The notebook can run on CPU but will be slower.


In [ ]:
!pip -q install "transformers>=4.48,<5" "accelerate>=1.0" "flask>=3,<4" "requests>=2.31"


## 2. Load the pretrained model

This version uses Qwen2.5-0.5B-Instruct. It is still small enough for Colab but follows structured productivity instructions more reliably than the previous FLAN-T5-small example.


In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

load_started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to("cpu")
model.eval()

print(f"Loaded: {MODEL_ID}")
print(f"Device: {DEVICE_NAME}")
print(f"Load time: {time.perf_counter() - load_started:.2f} seconds")


## 3. Define and test model behavior locally

The model is instructed to return a bounded proposal. The surrounding application remains responsible for validation, warnings, approval, and the final action.


In [ ]:
DEFAULT_INSTRUCTION = """
Extract proposed action items from the meeting notes.
Return a concise bullet list.
For every item include Task, Person, Deadline, and Evidence.
Use 'not stated' when information is missing.
Do not invent details.
Clearly mark uncertain assignments or deadlines.
""".strip()

SAMPLE_TEXT = """
Maya will send the revised outline by Friday.
Jordan may review the budget next week.
The group has not selected a presentation date.
""".strip()

def generate_text(text, instruction=DEFAULT_INSTRUCTION, max_new_tokens=220):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a bounded productivity assistant. Follow the requested format. "
                "Use only facts found in the supplied input."
            ),
        },
        {
            "role": "user",
            "content": f"{instruction}\n\nINPUT:\n{text}",
        },
    ]

    model_input = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    started = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(
            model_input,
            max_new_tokens=max(20, min(int(max_new_tokens), 256)),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_ids = generated_ids[0, model_input.shape[-1]:]
    output = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    elapsed_ms = round((time.perf_counter() - started) * 1000)
    return output, elapsed_ms

sample_output, sample_ms = generate_text(SAMPLE_TEXT)
print(sample_output)
print(f"\nLocal inference time: {sample_ms / 1000:.2f} seconds")

if not sample_output:
    raise RuntimeError("The local model test returned no text.")


## 4. Start the local API

This cell is restartable inside the updated notebook. It validates every request and provides three routes:

- `/`: simple status page
- `/health`: machine-readable connection test
- `/process`: authenticated model request


In [ ]:
import secrets
import threading
from flask import Flask, request, jsonify
from werkzeug.serving import make_server

PORT = 7860
SERVER_SECRET = secrets.token_urlsafe(24)
app = Flask(__name__)

@app.get("/")
def home():
    return (
        "<h1>Build 2 Colab Backend</h1>"
        "<p>The temporary model backend is running.</p>"
        "<p>Use the Apps Script connection test. Do not submit private data.</p>"
    )

@app.get("/health")
def health():
    return jsonify({
        "ok": True,
        "model": MODEL_ID,
        "device": DEVICE_NAME,
    })

@app.post("/process")
def process_text():
    if request.headers.get("X-Build-Secret") != SERVER_SECRET:
        return jsonify({"ok": False, "error": "Unauthorized request."}), 401

    data = request.get_json(silent=True) or {}
    text = str(data.get("text", "")).strip()
    instruction = str(data.get("instruction", "")).strip()

    if not text:
        return jsonify({"ok": False, "error": "Input cannot be empty."}), 400
    if len(text) > 5000:
        return jsonify({"ok": False, "error": "Input exceeds 5,000 characters."}), 400
    if not instruction:
        return jsonify({"ok": False, "error": "The model instruction is missing."}), 400
    if len(instruction) > 1500:
        return jsonify({"ok": False, "error": "The model instruction is too long."}), 400

    try:
        output, elapsed_ms = generate_text(
            text,
            instruction,
            data.get("max_new_tokens", 220),
        )
        if not output:
            return jsonify({"ok": False, "error": "The model returned an empty result."}), 502

        return jsonify({
            "ok": True,
            "output": output,
            "model": MODEL_ID,
            "device": DEVICE_NAME,
            "inference_ms": elapsed_ms,
        })
    except Exception as exc:
        print(f"Model error: {type(exc).__name__}: {exc}")
        return jsonify({"ok": False, "error": "The model could not complete the request."}), 500

if "http_server" in globals():
    try:
        http_server.shutdown()
    except Exception:
        pass

http_server = make_server("127.0.0.1", PORT, app)
server_thread = threading.Thread(target=http_server.serve_forever, daemon=True)
server_thread.start()
time.sleep(1)

import requests
local_health = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10)
local_health.raise_for_status()
print("Local API ready:", local_health.json())


## 5. Launch and verify the public backend

Run this one cell. It may take one or two minutes. It automatically retries temporary tunnel creation. It prints Apps Script configuration only after a real public model request succeeds.

If all attempts fail, restart the Colab session and use **Run all**. Do not copy an address from a failed attempt.


In [ ]:
import os
import re
import select
import socket
import subprocess
from urllib.parse import urlparse
from IPython.display import display, HTML

CLOUDFLARED_PATH = "/content/cloudflared"
CLOUDFLARED_DOWNLOAD = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64"
)

if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading the temporary tunnel client...")
    binary = requests.get(CLOUDFLARED_DOWNLOAD, timeout=180)
    binary.raise_for_status()
    with open(CLOUDFLARED_PATH, "wb") as file:
        file.write(binary.content)
    os.chmod(CLOUDFLARED_PATH, 0o755)

if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()

# Rotating the secret here ensures that only the final verified launch is used.
SERVER_SECRET = secrets.token_urlsafe(24)

def read_tunnel_url(process, timeout_seconds=45):
    deadline = time.time() + timeout_seconds
    captured = []
    while time.time() < deadline:
        if process.poll() is not None:
            break
        ready, _, _ = select.select([process.stdout], [], [], 1)
        if not ready:
            continue
        line = process.stdout.readline()
        if not line:
            continue
        captured.append(line.strip())
        match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if match:
            return match.group(0), captured
    return None, captured

def wait_until_public(base_url, process, timeout_seconds=90):
    hostname = urlparse(base_url).hostname
    deadline = time.time() + timeout_seconds
    last_error = "No response"

    while time.time() < deadline:
        if process.poll() is not None:
            return False, "The tunnel process stopped."
        try:
            socket.getaddrinfo(hostname, 443)
            response = requests.get(f"{base_url}/health", timeout=15)
            if response.status_code == 200 and response.json().get("ok"):
                return True, response.json()
            last_error = f"Health status {response.status_code}"
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        time.sleep(5)

    return False, last_error

PUBLIC_URL = None
tunnel_process = None
last_failure = None

for tunnel_attempt in range(1, 4):
    print(f"Tunnel attempt {tunnel_attempt}/3...")
    candidate_process = subprocess.Popen(
        [
            CLOUDFLARED_PATH,
            "tunnel",
            "--url", f"http://127.0.0.1:{PORT}",
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    candidate_url, tunnel_log = read_tunnel_url(candidate_process)
    if not candidate_url:
        last_failure = "The tunnel client did not produce an address."
        candidate_process.terminate()
        continue

    is_ready, readiness_result = wait_until_public(candidate_url, candidate_process)
    if not is_ready:
        last_failure = readiness_result
        candidate_process.terminate()
        try:
            candidate_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            candidate_process.kill()
        continue

    # A real authenticated model request must pass before configuration is shown.
    try:
        public_test = requests.post(
            f"{candidate_url}/process",
            headers={"X-Build-Secret": SERVER_SECRET},
            json={
                "text": SAMPLE_TEXT,
                "instruction": DEFAULT_INSTRUCTION,
                "max_new_tokens": 220,
            },
            timeout=180,
        )
        public_body = public_test.json()
        if public_test.status_code == 200 and public_body.get("ok") and public_body.get("output"):
            PUBLIC_URL = candidate_url
            tunnel_process = candidate_process
            VERIFIED_SAMPLE = public_body
            break
        last_failure = f"Public model test returned status {public_test.status_code}: {public_body}"
    except Exception as exc:
        last_failure = f"Public model test failed: {type(exc).__name__}: {exc}"

    candidate_process.terminate()

if not PUBLIC_URL:
    raise RuntimeError(
        "A verified public backend could not be created after three attempts. "
        f"Last result: {last_failure}. Restart the Colab session and select Run all."
    )

print("\n" + "=" * 72)
print("BACKEND READY — ALL AUTOMATIC TESTS PASSED")
print("=" * 72)
print("Copy only these two values into Code.gs:")
print(f"COLAB_API_BASE_URL: {PUBLIC_URL}")
print(f"COLAB_API_SECRET: {SERVER_SECRET}")
print("\nVerified model output:")
print(VERIFIED_SAMPLE["output"])
print(f"\nInference time: {VERIFIED_SAMPLE['inference_ms'] / 1000:.2f} seconds")
print("\nKeep this Colab runtime connected while using Apps Script.")

display(HTML(f"""
<div style="border:2px solid #16794d;border-radius:12px;padding:16px;background:#edf9f3;font-family:Arial,sans-serif">
  <strong style="color:#12613e">✓ BACKEND READY</strong>
  <p>All local and public tests passed. Copy the base URL and secret printed above into <code>Code.gs</code>.</p>
  <p>You do not need to open or edit the URL manually. Use the Apps Script <strong>Test connection</strong> button.</p>
</div>
"""))


## 6. Configure Apps Script

In `Code.gs`, replace exactly these two placeholders:

`COLAB_API_BASE_URL: 'PASTE_VERIFIED_BASE_URL_HERE'`

`COLAB_API_SECRET: 'PASTE_CURRENT_SECRET_HERE'`

Rules:

- Copy the base URL exactly as printed.
- Do not add `/health`.
- Do not add `/process`.
- Do not use an address from an earlier or failed run.
- Leave this Colab runtime connected.

Deploy the Apps Script web app and select **Test connection**. The Generate button stays disabled until Apps Script itself reaches this backend.


## 7. Required Build tests

1. **Successful input:** the model returns a useful proposal.
2. **Invalid input:** ordinary code blocks empty or insufficient input.
3. **Ambiguous input:** the app produces a warning and requires human review.
4. **Backend unavailable:** stop the tunnel after recording the other tests. The app must report that the backend is unavailable and take no final action.


## 8. Stop the backend

Run this only when you have finished testing and recording. You can also disconnect the Colab runtime.


In [ ]:
if "tunnel_process" in globals() and tunnel_process and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()
    print("Temporary public tunnel stopped.")
else:
    print("No active public tunnel was found.")

if "http_server" in globals():
    try:
        http_server.shutdown()
        print("Local API stopped.")
    except Exception:
        pass
